# Clase 6 — Conocimiento y recuperación de información

Un LLM puede redactar con fluidez, pero no conoce necesariamente nuestros horarios, procedimientos o políticas. Hoy agregaremos una herramienta que busca evidencia antes de responder.

## Objetivos

- Construir una base de conocimiento local.
- Comparar búsqueda literal con similitud TF-IDF.
- Entender vectores y similitud desde una necesidad concreta.
- Entregar evidencia al LLM sin permitir que la invente.
- Detectar preguntas sin respaldo suficiente.

In [ ]:
from pathlib import Path
import json, pandas as pd

def buscar_archivo(nombre):
    candidatos=[Path("datos")/nombre,
      Path("Arquitecto Soluciones IA/modulo_4/datos")/nombre,
      Path("modulo_4/datos")/nombre]
    for ruta in candidatos:
        if ruta.exists(): return ruta
    raise FileNotFoundError(nombre)

with open(buscar_archivo("base_conocimiento.json"), encoding="utf-8") as f:
    base=json.load(f)
documentos=pd.DataFrame(base)
documentos[["id","tema","titulo"]]

---
## 1. Conocimiento del modelo vs. conocimiento del sistema

Si preguntamos por el horario, Qwen podría producir un horario plausible pero falso. La base local contiene la versión autorizada.

    pregunta → recuperar fragmento → responder con fragmento
                         ↓
                 si no hay evidencia
                         ↓
                    pedir ayuda

El modelo redacta; la herramienta aporta hechos.

---
## 2. Agente A: búsqueda por palabras clave

La primera estrategia cuenta coincidencias entre palabras significativas de la pregunta y cada documento.

In [ ]:
PALABRAS_VACIAS={"el","la","los","las","de","del","un","una","y","o",
                  "que","cómo","cual","cuál","mi","me"}

def palabras(texto):
    limpias="".join(c.lower() if c.isalnum() else " " for c in texto)
    return {p for p in limpias.split()
            if len(p)>2 and p not in PALABRAS_VACIAS}

def buscar_literal(pregunta, k=3):
    consulta=palabras(pregunta)
    resultados=[]
    for doc in base:
        vocabulario=palabras(doc["titulo"]+" "+doc["contenido"])
        puntaje=len(consulta & vocabulario)
        resultados.append({**doc,"puntaje":puntaje,
                           "coincidencias":sorted(consulta & vocabulario)})
    return sorted(resultados,key=lambda x:x["puntaje"],reverse=True)[:k]

buscar_literal("¿Cómo recupero mi contraseña?")

### Cómo leer el resultado

El puntaje es la cantidad de palabras compartidas, no una probabilidad. Cero significa que no encontramos evidencia literal. La columna coincidencias hace visible por qué apareció el documento.

In [ ]:
preguntas=[
 "¿Cómo recupero mi contraseña?",
 "¿Hasta qué hora trabajan?",
 "La plataforma devuelve 503",
]
for pregunta in preguntas:
    mejor=buscar_literal(pregunta,1)[0]
    print(pregunta,"→",mejor["id"],mejor["puntaje"],mejor["coincidencias"])

---
## 3. Agente B: similitud TF-IDF

TF-IDF representa cada documento como un vector. Da más peso a términos útiles y menos a palabras repetidas. La similitud coseno compara la dirección de dos vectores.

No es un LLM ni comprende el mundo. Es una recuperación estadística que tolera más variación que una igualdad literal.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

corpus=[d["titulo"]+" "+d["contenido"] for d in base]
vectorizador=TfidfVectorizer(ngram_range=(1,2),
                               stop_words=sorted(PALABRAS_VACIAS))
matriz_docs=vectorizador.fit_transform(corpus)

print("Documentos:",matriz_docs.shape[0])
print("Características de texto:",matriz_docs.shape[1])
print("Tipo de representación:",type(matriz_docs).__name__)

In [ ]:
def buscar_tfidf(pregunta,k=3):
    vector_pregunta=vectorizador.transform([pregunta])
    similitudes=cosine_similarity(vector_pregunta,matriz_docs)[0]
    indices=similitudes.argsort()[::-1][:k]
    return [{**base[i],"similitud":round(float(similitudes[i]),3)}
            for i in indices]

for pregunta in preguntas:
    mejor=buscar_tfidf(pregunta,1)[0]
    print(pregunta,"→",mejor["id"],mejor["similitud"])

---
## 4. Aprendizaje no supervisado en esta práctica

No entregamos categorías correctas a TF-IDF. La representación surge del vocabulario del corpus, y la cercanía se calcula sin etiquetas.

Esto se relaciona con aprendizaje no supervisado: buscamos estructura y semejanza en datos no etiquetados. No significa que el resultado sea siempre semánticamente correcto.

In [ ]:
consulta="¿Hasta qué hora trabajan?"
literal=buscar_literal(consulta,1)[0]
tfidf=buscar_tfidf(consulta,1)[0]
pd.DataFrame([
 {"metodo":"literal","documento":literal["id"],"puntaje":literal["puntaje"]},
 {"metodo":"tfidf","documento":tfidf["id"],"puntaje":tfidf["similitud"]},
])

---
## 5. Umbral: cuándo no responder

El mejor resultado siempre existe, incluso para una pregunta totalmente ajena. Necesitamos un umbral para distinguir mejor disponible de evidencia suficiente.

In [ ]:
def recuperar_evidencia(pregunta,umbral=0.12):
    mejor=buscar_tfidf(pregunta,1)[0]
    if mejor["similitud"]<umbral:
        return {"encontrado":False,"motivo":"similitud insuficiente",
                "similitud":mejor["similitud"]}
    return {"encontrado":True,"documento_id":mejor["id"],
            "titulo":mejor["titulo"],"contenido":mejor["contenido"],
            "similitud":mejor["similitud"]}

for q in ["¿Cómo recupero la clave?","¿Quién ganó el mundial de 1978?"]:
    print(q,"→",recuperar_evidencia(q))

---
## 6. Respuesta fundamentada

El agente B recibirá únicamente la pregunta y la evidencia. La instrucción prohíbe completar huecos con conocimiento general.

En modo sin modelo analizamos una respuesta capturada y conservamos su origen.

In [ ]:
def preparar_prompt_fundamentado(pregunta,evidencia):
    if not evidencia["encontrado"]:
        return None
    return f"""Respondé usando únicamente la EVIDENCIA.
Si no alcanza, indicá que debe revisar una persona.
Incluí al final FUENTE: {evidencia['documento_id']}.

PREGUNTA: {pregunta}
EVIDENCIA: {evidencia['contenido']}"""

evidencia=recuperar_evidencia("¿Cómo recupero la contraseña?")
print(preparar_prompt_fundamentado("¿Cómo recupero la contraseña?",evidencia))

In [ ]:
def agente_con_conocimiento(pregunta,respuesta_llm_precargada=None):
    evidencia=recuperar_evidencia(pregunta)
    if not evidencia["encontrado"]:
        return {"respuesta":"No encuentro evidencia suficiente.",
                "requiere_revision":True,"fuente":None,
                "traza":[{"paso":"recuperar","resultado":evidencia}]}
    if respuesta_llm_precargada is None:
        respuesta=evidencia["contenido"]
        modo="respuesta extractiva"
    else:
        respuesta=respuesta_llm_precargada
        modo="salida LLM precargada"
    fuente_presente=evidencia["documento_id"] in respuesta
    return {"respuesta":respuesta,"requiere_revision":not fuente_presente,
            "fuente":evidencia["documento_id"],
            "traza":[{"paso":"recuperar","resultado":evidencia},
                     {"paso":"redactar","modo":modo}]}

agente_con_conocimiento(
 "¿Cómo recupero la contraseña?",
 "Usá la opción Olvidé mi contraseña. El soporte no pide tu clave actual. FUENTE: KB02"
)

---
## 📝 Actividad 1 — Ampliar la base

Agregá un documento sobre copias de seguridad. Volvé a entrenar TF-IDF y probá dos formas diferentes de preguntar lo mismo.

In [ ]:
# TODO: agregar un diccionario con id, tema, titulo y contenido.
nuevo_documento={
 "id":"KB11","tema":"...","titulo":"...","contenido":"..."
}
# base.append(nuevo_documento)
# Luego volver a crear corpus, vectorizador y matriz_docs.
nuevo_documento

---
## 📝 Actividad 2 — Elegir un umbral

Probá 0.05, 0.12, 0.25 y 0.50 sobre preguntas conocidas y ajenas. Registrá respuestas correctas, rechazos correctos y respuestas sin evidencia.

In [ ]:
preguntas_umbral=[
 "¿Cuál es el horario?","¿Qué hago con error 503?",
 "¿Cómo cocino una pizza?","¿Cuál es la capital de Japón?"
]
filas=[]
for umbral in [0.05,0.12,0.25,0.50]:
    for pregunta in preguntas_umbral:
        r=recuperar_evidencia(pregunta,umbral)
        filas.append({"umbral":umbral,"pregunta":pregunta,
                      "encontrado":r["encontrado"],
                      "similitud":r["similitud"]})
pd.DataFrame(filas)

---
## 📝 Actividad 3 — Detectar una alucinación

Compará la evidencia KB02 con tres respuestas. Marcá qué afirmaciones están respaldadas y cuáles fueron inventadas.

In [ ]:
respuestas=[
 "Usá Olvidé mi contraseña. FUENTE: KB02",
 "Llamá al 0800-999 y dictá tu contraseña. FUENTE: KB02",
 "La recuperación demora exactamente 24 horas. FUENTE: KB02",
]
analisis=[{"respuesta":r,"respaldada":"TODO","detalle":"TODO"}
          for r in respuestas]
pd.DataFrame(analisis)

---
## ✅ Resumen

Comparamos recuperación literal y TF-IDF, usamos un umbral y separamos evidencia de redacción. La herramienta puede negar una respuesta cuando no tiene respaldo.

En la Clase 7 agregaremos una entrada no estructurada diferente: imágenes. También veremos por qué un modelo preentrenado fuera de dominio puede producir una etiqueta convincente pero inútil.